# LangChain中的短期记忆（InMemorySaver）

本文基于 `langchain==1.3.0`，只聚焦一个主题：**Agent 的记忆，尤其是短期记忆（short-term memory）**。

主要内容：
- 记忆有哪些常见分类，分别是什么意思
- 短期记忆和长期记忆的区别，以及各自适用场景
- LangChain / LangGraph 中的短期记忆为什么依赖 `Checkpointer`
- `thread_id`、checkpoint、时间回缩（time travel）之间是什么关系

## 一、先把记忆这件事讲清楚

在 Agent 场景里，`memory` 的核心作用只有一句话：**让系统能够记住之前发生过的事，并在后续继续利用这些信息。**
从作用范围上可以做如下分类：

- `short-term memory`：短期记忆，只在**单个会话 / 单个 thread** 内生效
- `long-term memory`：长期记忆，可以**跨会话、跨 thread** 持续存在

这是最重要的一组分类，因为它直接决定你该把数据存在哪里。


## 二、短期记忆和长期记忆怎么选

一个简单判断方法：**这条信息是不是只对当前会话有意义？**

如果是，就更适合放短期记忆；如果希望下次开新会话还能拿到，就更适合放长期记忆。

| 类型 | 作用范围 | 典型内容 | 什么时候用 | 常见实现 |
| --- | --- | --- | --- | --- |
| 短期记忆 | 单个 `thread` | 当前消息历史、当前上传文件、当前推理中间状态 | 需要连续多轮对话时 | `AgentState` + `Checkpointer` |
| 长期记忆 | 跨 `thread` | 用户画像、偏好、历史事实、业务知识 | 需要跨会话复用时 | `Store` |

可以把它们理解成：
- 短期记忆解决“**这次对话别忘了前文**”
- 长期记忆解决“**下次再来还能认出你**”

两者不是互斥关系，真实项目里经常同时存在。


## 三、短期记忆的核心：AgentState + Checkpointer

在 LangChain 中，短期记忆本质上是 **Agent 的状态（`AgentState`）**。

默认情况下，这个状态里最重要的字段是 `messages`，也就是消息历史。但它不一定只包含消息，还可以包含：
- 当前用户 id
- 当前会话里的偏好设置
- 检索结果
- 本轮任务产生的中间结果

那为什么还需要 `Checkpointer`？

因为 Agent 不只是“临时拿着一份状态跑完就算了”，而是要把运行过程中的状态**保存成快照**。LangChain 官方的做法是：把状态交给 `Checkpointer` 持久化。在一次完整执行里，通常会生成一个或多个 checkpoint，具体数量取决于实际执行过程。

![短期记忆与时间回缩示意图](assets/09-memory-time-travel-check-1.jpg)

这样带来的好处有 3 个：
- 下一轮对话开始时，可以把上一个状态继续接上
- 同一个会话里的多次交互，不会互相丢上下文
- 可以查看历史快照，甚至回到某个旧快照继续执行

补一句：`InMemorySaver` 来自 `langgraph`，不是 `langchain`。原因是 `Checkpointer`、状态快照、时间回缩这类能力，本质上更接近 **Agent 运行图（graph）层** 的能力；而 LangChain 现在很多 Agent 能力本身就是构建在 LangGraph 之上的，所以这类组件放在 `langgraph.checkpoint` 下面会更合理。


In [ ]:
# 初始化一个支持工具调用的聊天模型
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver

# 加载 .env 中的环境变量，便于读取 API Key
load_dotenv()

# 创建本篇示例要使用的聊天模型
model = ChatOpenAI(
    model="qwen3.5-122b-a10b",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0,
)

# 自定义状态：除了 messages，再额外保存 user_id 和 preferences
# user_id：用来标识当前用户是谁。真实项目里，后续工具调用、查数据库、查用户资料时，经常要靠它定位用户。
# preferences：用来保存当前用户的一些偏好设置。比如这里的 {"style": "concise"} 就表示“回答尽量简洁”。
# 注意：这里只是演示“这些字段会被保存进状态”，并不代表模型会自动读取它们并改变回答风格。
class CustomAgentState(AgentState):
    user_id: str
    preferences: dict

# 用内存版 Checkpointer 保存 AgentState 快照，适合本地演示
checkpointer = InMemorySaver()

# 创建带短期记忆能力的 Agent：状态结构由 state_schema 定义，状态持久化由 checkpointer 负责
agent = create_agent(
    model=model,
    tools=[],
    state_schema=CustomAgentState,
    checkpointer=checkpointer,
)


In [ ]:
# 同一个 thread_id 代表同一组短期记忆，其实就是同一个会话id
config = {"configurable": {"thread_id": "demo-thread-001"}}

# 第 1 轮：告诉模型我的名字
agent.invoke(
    {
        "messages": [{"role": "user", "content": "你好，我叫 Bob。"}],
        "user_id": "u_001",
        "preferences": {"style": "concise"},
    },
    config,
)

# 第 2 轮：沿用同一个 thread_id，Agent 可以接着上次的短期记忆继续
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "你还记得我叫什么吗？"}],
        "user_id": "u_001",
        "preferences": {"style": "concise"},
    },
    config,
)
# 第 3 轮：沿用同一个 thread_id，Agent 可以接着上次的短期记忆继续
result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "我喜欢的颜色是蓝色。"}],
        "user_id": "u_001",
        "preferences": {"style": "concise"},
    },
    config,
)

# 取最后一条消息，通常就是 Agent 给用户的最新回复
print(result["messages"][-1].content)


如果换一个 thread_id，就相当于开启了另一段会话，比如：

In [ ]:
# 更换thread_id
new_config = {"configurable": {"thread_id": "demo-thread-002"}}

# 用新的 thread_id 再问一次，相当于在新会话里发问
new_result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "你还记得我叫什么吗？"}],
        "user_id": "u_001",
        "preferences": {"style": "concise"},
    },
    new_config,
)

# 对比这个结果：它通常不会继承 demo-thread-001 里的名字记忆
print(new_result["messages"][-1].content)


## 四、`thread_id`、checkpoint、会话到底是什么关系

`thread_id` 可以理解成：**一条会话线索的标识符**。

它很像“会话 id”，但比传统聊天系统里的 session 更准确一点，因为 LangChain / LangGraph 用它来串起整条状态时间线。

也就是说：
- 同一个 `thread_id` 下，会累积出多个 checkpoint
- 这些 checkpoint 共同组成这条会话的历史
- 不同 `thread_id` 之间，短期记忆天然隔离

所以你可以把关系理解成：

`thread_id` = 会话线 id

`checkpoint` = 这条会话线上的某一个历史快照

换句话说，`thread_id` 不是操作系统里的线程 id，而更接近**会话 id / 对话 id / conversation id**。


In [ ]:
# 查看当前 thread 的状态历史
# 读取该 thread_id 下的全部 checkpoint 历史
history = list(agent.get_state_history(config))

# 最新 checkpoint 在最前面，先拿一个看结构
# 取最新一个快照，观察它保存了哪些状态信息
latest = history[0]

# checkpoint_id 是单个快照的唯一标识；thread_id 更稳妥的读取方式是从 configurable 里获取
print("最新 checkpoint_id:", latest.config["configurable"]["checkpoint_id"])
print("所属 thread_id:", latest.config["configurable"]["thread_id"])
# values 里保存的是这个时间点上的完整状态，这里只演示查看 messages 数量
print("当前 messages 条数:", len(latest.values["messages"]))


## 五、时间回缩（time travel）是什么

时间回缩的本质是：**回到某个旧 checkpoint，再从那个时间点继续运行。**

它不是“把历史删除掉”，而更像：
- 先定位到某个历史快照
- 用那个快照作为新的起点
- 再继续发消息、继续执行 Agent

它很适合做 3 类事：
- 调试 Agent：看某一步之前的状态到底是什么
- 人工纠错：发现某轮走偏了，从更早的节点重新来
- 分支实验：同一段历史，尝试不同后续输入，观察结果差异

从实现角度看，关键就是拿到历史里的 `checkpoint_id`，然后把它放回 `configurable` 配置里。


In [ ]:
# 先简单查看每个 checkpoint 里最后一条用户消息是什么
for i, state in enumerate(history[:8]):
    messages = state.values["messages"]
    last_message = messages[-1].content if messages else "空"
    print(i, state.config["configurable"]["checkpoint_id"], last_message)

# 说明：history[0] 是最新快照，索引越大表示越早的快照
# 选择回溯点时，优先选“目标信息还没出现，且离现在最近”的那个 checkpoint
print("说明：history[0] 是最新快照，索引越大越早。")
print("回溯时，选目标信息还没出现、且离现在最近的那个 checkpoint。")
print("另外，如果重复执行同一个 thread_id，对应的 history 还会继续累积。")

现在要做的是，让 checkpoint 回溯到“我说出喜欢蓝色之前”的状态，流程大致如下。

注意：下面的 `history[2]` 只是**当前这次示例运行结果**里恰好合适的回溯点，并不是固定规律。实际使用时，应该先看上面打印出来的 history，再决定选哪个 checkpoint。

![短期记忆与时间回缩示意图](assets/09-memory-time-travel-check-2.jpg)

In [ ]:
# 这里选择 history[2]，因为在当前这次运行里，它正好对应“还没告诉模型喜欢蓝色”之前的状态
old_checkpoint_id = history[2].config["configurable"]["checkpoint_id"]

# 在原有 thread_id 的基础上额外指定 checkpoint_id，就能定位到某个历史快照
rollback_config = {
    "configurable": {
        "thread_id": "demo-thread-001",
        "checkpoint_id": old_checkpoint_id,
    }
}

# 从旧 checkpoint 继续执行：由于那时还没说过喜欢蓝色，所以模型通常答不上来
rollback_result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "我喜欢什么颜色？"}
        ],
        "user_id": "u_001",
        "preferences": {"style": "concise"},
    },
    rollback_config,
)

# 打印基于旧快照继续执行后的结果
print(rollback_result["messages"][-1].content)


说明：

真实产品里的前端，通常不会把 checkpoint_id 和底层快照列表直接暴露给普通用户，
前端里更常见的是这几种实现：

“编辑并重发”
用户改一条旧消息，系统从那条消息对应的状态重新生成后续内容。底层本质上还是回到某个历史状态再继续跑，只是前端不会显示 checkpoint_id。

“回到这里” / “从这里继续”
在某条消息旁边放一个按钮。用户点击后，前端把这条消息对应的内部状态 id 传给后端，后端再基于那个状态继续执行。这里用户看到的是“某条消息”或“某个对话节点”，不是 checkpoint 原始 id。

分支对话
回溯后不是覆盖原对话，而是新开一个分支。前端上看起来像“从这条消息开始继续聊出另一条线”。

## 六、最后总结

把这篇内容压缩成几句话，就是：

- 记忆先分短期和长期，前者解决当前会话连续性，后者解决跨会话复用
- LangChain 的短期记忆，本质是 `AgentState`
- `Checkpointer` 负责把状态保存成一个个 checkpoint 快照
- 同一个 `thread_id` 下会形成一组 checkpoint，它表示同一条会话线
- 时间回缩就是回到某个旧 checkpoint，再从那里继续执行

如果只记一个最关键的句子，那就是：**短期记忆 = 某个 `thread_id` 下持续演化的一组 AgentState 快照。**
